# FlexRank Hub Load, Save, and Prune

A compact workflow for loading a self-contained FlexRank causal language model from the Hugging Face Hub, evaluating its stored profiles, deploying a physically pruned GAR checkpoint, and verifying that the reloaded checkpoint preserves loss and storage metadata.

The default configuration uses `riccardozaccone96/flexrank-llama3.2-1B`. For the optional local DataSVD/profile-creation path, set `FLEXRANK_CHECKPOINT` to `None` and change `MODEL_NAME_OR_PATH` to an ordinary base-model checkpoint.


## Imports and setup

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "flextrain").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

from datasets import Dataset, IterableDataset
from transformers import TrainingArguments

from flexrank.layers import ODImpl, ODLinear
from flexrank.profiles import DPSearchAlgo
from flexrank.samplers.base_sampler import DeployMode
from flexrank.trainers import SVDType
from flextrain.dataset import load_dataset_from_hf
from flextrain.model import FlexRankModel, ModelWithProcessorAndMetric, load_model_from_hf, replace_conv1d_with_linear
from flextrain.utils.args import NLPDataArguments, NLPModelArguments, TaskName
from flextrain.utils.flexrank_utils import init_eval_trainer, init_flexrank_model
from flextrain.utils.utils import suppress_stdout


class ExportNamespace(SimpleNamespace):
    def to_export_dict(self):
        return dict(vars(self))


def materialize_dataset(dataset):
    if isinstance(dataset, IterableDataset):
        return Dataset.from_list(list(dataset))
    return dataset

@suppress_stdout
def evaluate_model(model, eval_dataset, collator, processor):
    eval_trainer = init_eval_trainer(
        ModelWithProcessorAndMetric(model, processor, None),
        collator,
        eval_dataset,
        eval_args,
        deepcopy_model=False,
    )
    return eval_trainer.evaluate()


def extract_eval_loss(metrics):
    if "eval_loss" not in metrics:
        raise RuntimeError(f"Evaluation did not return eval_loss. Available metrics: {sorted(metrics)}")
    return float(metrics["eval_loss"])


def decomposed_tensor_bytes(model, layer_names, prefix=""):
    """Return tensor payload bytes without copying model weights."""
    return sum(
        tensor.numel() * tensor.element_size()
        for key, tensor in model.state_dict().items()
        if any(key.startswith(f"{prefix}{name}.") for name in layer_names)
    )

def odlinear_impl_counts(model):
    counts = {impl.value: 0 for impl in ODImpl}
    for module in model.modules():
        if isinstance(module, ODLinear):
            counts[module.impl.value] += 1
    return {key: value for key, value in counts.items() if value}



Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [ ]:
# Model and dataset
MODEL_NAME_OR_PATH = "riccardozaccone96/flexrank-llama3.2-1B"  #@param {type:"string"}
EXCLUDE_LAYERS_NAMES = "lm_head"  #@param {type:"string"}

DATASET_PATH = "HuggingFaceFW/fineweb-edu"  #@param {type:"string"}
DATASET_NAME = "sample-10BT"  #@param {type:"string"}
SEQUENCE_LENGTH = 1024  #@param {type:"integer"}
EVAL_DS_SIZE = 4096  #@param {type:"integer"}
CALIB_DS_SIZE = 256  #@param {type:"integer"}

# Pruning and profiles
TEST_SIZE_RATIOS = "0.45, 0.65, 0.95"  #@param {type:"string"}
PROFILE_N_MODELS = 10  #@param {type:"integer"}
PROFILE_MIN_P = 0.1  #@param {type:"number"}
DEPLOY_MODE = "GAR"  #@param ["NO", "SVD", "GAR"]

# Hub/local checkpoint controls
# For local DataSVD, set this to None and MODEL_NAME_OR_PATH to a base model.
FLEXRANK_CHECKPOINT = "riccardozaccone96/flexrank-llama3.2-1B"  #@param {type:"string"}

# Normalize form-like strings into the values used by later cells.
MODEL_TAG = MODEL_NAME_OR_PATH.split("/")[-1].replace("-", "_")
EXCLUDE_LAYERS_NAMES = [item.strip() for item in EXCLUDE_LAYERS_NAMES.split(",") if item.strip()]
DATASET_NAME = DATASET_NAME or None
TEST_SIZE_RATIOS = tuple(float(item.strip()) for item in TEST_SIZE_RATIOS.split(",") if item.strip())
DEPLOY_SIZE_RATIO = TEST_SIZE_RATIOS[-1]
DEPLOY_MODE = {mode.name: mode for mode in DeployMode}[DEPLOY_MODE.upper()]
FLEXRANK_CHECKPOINT = FLEXRANK_CHECKPOINT or None

# Saving directories
OUTPUT_ROOT = Path("outputs/notebooks")
BASE_SAVE_DIR = OUTPUT_ROOT / f"{MODEL_TAG}_base_demo"
SAVE_DIR = OUTPUT_ROOT / f"{MODEL_TAG}_flexrank_demo"
EVAL_DIR = OUTPUT_ROOT / f"{MODEL_TAG}_flexrank_eval"
PRUNED_SAVE_DIR = SAVE_DIR.parent / f"{SAVE_DIR.name}_{DEPLOY_MODE.value}_{DEPLOY_SIZE_RATIO:g}"

{
    "model": MODEL_NAME_OR_PATH,
    "flexrank_checkpoint": FLEXRANK_CHECKPOINT,
    "test_size_ratios": TEST_SIZE_RATIOS,
    "deployment_size_ratio": DEPLOY_SIZE_RATIO,
    "deploy_mode": DEPLOY_MODE.name,
    "save_dir": str(SAVE_DIR),
}


{'model': 'riccardozaccone96/flexrank-llama3.2-1B',
 'flexrank_checkpoint': 'riccardozaccone96/flexrank-llama3.2-1B',
 'test_size_ratios': (0.45, 0.65, 0.95),
 'deployment_size_ratio': 0.95,
 'deploy_mode': 'GAR',
 'save_dir': 'outputs/notebooks/flexrank_llama3.2_1B_flexrank_demo'}

## Obtain a FlexRankModel

### Load the Hub checkpoint or build locally

By default, the next cell loads the uploaded FlexRank model and tokenizer directly from the Hub. For local DataSVD/profile creation, set `FLEXRANK_CHECKPOINT = None` and point `MODEL_NAME_OR_PATH` to an ordinary base model.


Load the model, tokenizer, and a small tokenized FineWeb-Edu split through the same Hugging Face helpers used by `train.py`. `trust_remote_code=True` loads the FlexRank implementation stored with the Hub checkpoint.


In [ ]:
model_args = NLPModelArguments(
    model_name_or_path=MODEL_NAME_OR_PATH,
    trust_remote_code=True,
    use_auth_token=True,
    load_pretrained_model=True,
    _torch_dtype="torch.float32",
    cache_dir=None,
)
model_data = load_model_from_hf(TaskName.NLP, model_args)
replace_conv1d_with_linear(model_data.model)

data_args = NLPDataArguments(
    path=DATASET_PATH,
    name=DATASET_NAME,
    streaming=True,
    eval_ds_size=EVAL_DS_SIZE,
    calib_ds_size=CALIB_DS_SIZE,
    sequence_length=SEQUENCE_LENGTH,
    cache_file_names=None,
)
splits = load_dataset_from_hf(TaskName.NLP, data_args, model_data.proc, data_seed=0)
splits = SimpleNamespace(
    train=splits.train,
    val=materialize_dataset(splits.val),
    calib=materialize_dataset(splits.calib),
    collator=splits.collator,
)

eval_args = TrainingArguments(
    output_dir=EVAL_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=16,
    max_steps=1,
    learning_rate=0.0,
    report_to="none",
    disable_tqdm=True,
    remove_unused_columns=False,
    label_names=["labels"],
)

base_metrics = evaluate_model(model_data.model, splits.val, splits.collator, model_data.proc)

if FLEXRANK_CHECKPOINT is None:
    shutil.rmtree(BASE_SAVE_DIR, ignore_errors=True)
    model_data.model.save_pretrained(BASE_SAVE_DIR)

base_metrics

Initialize the FlexRank model by decomposing the source model with DataSVD on the calibration tokens, then wrap the decomposed model as `FlexRankModel`.


In [4]:
if FLEXRANK_CHECKPOINT is None:
    calib_trainer = init_eval_trainer(
        model_data,
        splits.collator,
        splits.calib,
        eval_args,
        deepcopy_model=False,
    )
    demo_args = SimpleNamespace(
        dataset=ExportNamespace(**data_args.to_export_dict()),
        decomposition=ExportNamespace(
            svd_type=SVDType.DataSVD,
            exclude_layers_names=EXCLUDE_LAYERS_NAMES,
            freeze_non_decomposed=False,
            max_svd_data_count=CALIB_DS_SIZE,
            gram_cache_device="cpu",
        ),
        profile_algo=ExportNamespace(classname="DPSearchAlgo"),
        sampler=ExportNamespace(classname="BaseSampler"),
        train=ExportNamespace(freeze_non_decomposed=False),
        model=ExportNamespace(**model_args.to_export_dict()),
    )

    flex_model_data, decomposed_params = init_flexrank_model(
        model_data,
        calib_trainer,
        demo_args,
    )
    flex_model = flex_model_data.model
else:
    decomposed_params = None
    # `model_data.model` was already loaded from the Hub with its tokenizer.
    flex_model = model_data.model

flex_metrics = (
    base_metrics
    if FLEXRANK_CHECKPOINT
    else evaluate_model(flex_model, splits.val, splits.collator, model_data.proc)
)

{
    "source": "hub" if FLEXRANK_CHECKPOINT else "local_datasvd",
    "decomposed_layers": len(flex_model.decomposed_layer_names),
    "decomposed_params": decomposed_params,
    "sample_layers": sorted(flex_model.decomposed_layer_names)[:5],
    **flex_metrics,
}


{'source': 'hub',
 'decomposed_layers': 112,
 'decomposed_params': None,
 'sample_layers': ['model.layers.0.mlp.down_proj',
  'model.layers.0.mlp.gate_proj',
  'model.layers.0.mlp.up_proj',
  'model.layers.0.self_attn.k_proj',
  'model.layers.0.self_attn.o_proj'],
 'eval_loss': 2.588521957397461,
 'eval_runtime': 700.6456,
 'eval_samples_per_second': 5.846,
 'eval_steps_per_second': 0.365,
 'epoch': 0.00048828125}

Model pruning (via `flex_model.reduce_size(...)`) requires profile metadata to select the proper submodel. For this compact demo, use `DPSearchAlgo` to create budget-aware profiles. In full experiments, run the Knowledge Consolidation phase to obtain a performant elastic model.

In [5]:
if FLEXRANK_CHECKPOINT is None:
    profile_trainer = init_eval_trainer(
        ModelWithProcessorAndMetric(flex_model.base_model, model_data.proc, None),
        splits.collator,
        splits.calib,
        eval_args,
        deepcopy_model=False,
    )

    dp_solution = DPSearchAlgo(
        evaluator=profile_trainer,
        n_models=PROFILE_N_MODELS,
        min_p=PROFILE_MIN_P
    ).solve()

    flex_model.profiles_data = dp_solution.to_profiles_data()


In [6]:
if FLEXRANK_CHECKPOINT is None:
    shutil.rmtree(SAVE_DIR, ignore_errors=True)
    flex_model.save_pretrained(SAVE_DIR)


### Use the Hub-loaded model or reload the local checkpoint


In [8]:
CHECKPOINT_PATH = FLEXRANK_CHECKPOINT or SAVE_DIR

loaded_model = (
    flex_model
    if FLEXRANK_CHECKPOINT
    else FlexRankModel.from_pretrained(CHECKPOINT_PATH)
)
loaded_metrics = (
    flex_metrics
    if FLEXRANK_CHECKPOINT
    else evaluate_model(loaded_model, splits.val, splits.collator, model_data.proc)
)

reference_layer_names = loaded_model.decomposed_layer_names
if FLEXRANK_CHECKPOINT:
    reference_tensor_bytes = decomposed_tensor_bytes(
        loaded_model,
        reference_layer_names,
        prefix="_wrapped_model.",
    )
    storage_reference = "full FlexRank tensor payload"
else:
    reference_tensor_bytes = decomposed_tensor_bytes(
        model_data.model,
        reference_layer_names,
    )
    storage_reference = "dense base-model tensor payload"

{
    "source": "hub" if FLEXRANK_CHECKPOINT else "local_datasvd",
    "checkpoint": str(CHECKPOINT_PATH),
    "base_model_type": loaded_model.config.base_model_type,
    "base_auto_class": loaded_model.config.base_auto_class,
    "decomposed_layers": len(loaded_model.decomposed_layer_names),
    "storage_reference": storage_reference,
    **loaded_metrics,
}


{'source': 'hub',
 'checkpoint': 'riccardozaccone96/flexrank-llama3.2-1B',
 'base_model_type': 'llama',
 'base_auto_class': 'AutoModelForCausalLM',
 'decomposed_layers': 112,
 'storage_reference': 'full FlexRank tensor payload',
 'eval_loss': 2.588521957397461,
 'eval_runtime': 700.6456,
 'eval_samples_per_second': 5.846,
 'eval_steps_per_second': 0.365,
 'epoch': 0.00048828125}

## Pruning the model

### Evaluating different submodels without physical pruning

In [8]:
pruned_eval_rows = []
for size_ratio in TEST_SIZE_RATIOS:
    loaded_model.reduce_size(size_ratio=size_ratio)
    metrics = evaluate_model(loaded_model, splits.val, splits.collator, model_data.proc)
    pruned_eval_rows.append(
        {
            "target_size_ratio": size_ratio,
            "virtual_size_ratio": loaded_model.virtual_size_ratio,
            "physical_size_ratio": loaded_model.physical_size_ratio,
            "eval_loss": extract_eval_loss(metrics),
        }
    )

pruned_eval_rows

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[{'target_size_ratio': 0.45,
  'virtual_size_ratio': 0.44906376353625593,
  'physical_size_ratio': 1.0,
  'eval_loss': 2.7562198638916016},
 {'target_size_ratio': 0.65,
  'virtual_size_ratio': 0.6470904925773884,
  'physical_size_ratio': 1.0,
  'eval_loss': 2.6522269248962402},
 {'target_size_ratio': 0.95,
  'virtual_size_ratio': 0.947857468292631,
  'physical_size_ratio': 1.0,
  'eval_loss': 2.6088428497314453}]

### Deploying a model
Prune to a requested size and physically deploy the decomposed weights. `DeployMode.NO` ("NO") keeps the profile virtual, `DeployMode.SVD` ("SVD") stores sliced SVD factors, and `DeployMode.GAR` ("GAR") stores GAR factors where supported; unsupported layers fall back to SVD with a warning.

In [10]:
loaded_model.reduce_size(size_ratio=DEPLOY_SIZE_RATIO, deploy_mode=DEPLOY_MODE)
deploy_metrics = evaluate_model(loaded_model, splits.val, splits.collator, model_data.proc)

{
    "target_size_ratio": DEPLOY_SIZE_RATIO,
    "virtual_size_ratio": loaded_model.virtual_size_ratio,
    "physical_size_ratio": loaded_model.physical_size_ratio,
    "eval_loss": extract_eval_loss(deploy_metrics),
    "odlinear_impl_counts": odlinear_impl_counts(loaded_model),
}

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'target_size_ratio': 0.95,
 'virtual_size_ratio': 0.947857468292631,
 'physical_size_ratio': 0.947857468292631,
 'eval_loss': 2.608842611312866,
 'odlinear_impl_counts': {'gar': 112}}

### Saving the pruned model
Save the physically pruned model, compare its decomposed tensor payload with the unpruned Hub model (or dense base model for the local DataSVD path), without copying weights, then reload it to verify loss and storage metadata.


In [10]:
shutil.rmtree(PRUNED_SAVE_DIR, ignore_errors=True)
loaded_model.save_pretrained(PRUNED_SAVE_DIR)

reloaded_pruned_model = FlexRankModel.from_pretrained(PRUNED_SAVE_DIR)
reloaded_pruned_metrics = evaluate_model(reloaded_pruned_model, splits.val, splits.collator, model_data.proc)

layer_names = reloaded_pruned_model.decomposed_layer_names
pruned_tensor_bytes = decomposed_tensor_bytes(
    reloaded_pruned_model,
    layer_names,
    prefix="_wrapped_model.",
)
storage_size_ratio = pruned_tensor_bytes / reference_tensor_bytes

{
    "save_dir": str(PRUNED_SAVE_DIR),
    "physical_size_ratio": reloaded_pruned_model.physical_size_ratio,
    "storage_reference": storage_reference,
    "decomposed_storage_size_ratio_vs_reference": storage_size_ratio,
    "storage_vs_internal_delta": storage_size_ratio - reloaded_pruned_model.physical_size_ratio,
    "deployed_eval_loss": extract_eval_loss(deploy_metrics),
    "reloaded_eval_loss": extract_eval_loss(reloaded_pruned_metrics),
}


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'save_dir': 'outputs/notebooks/flexrank_llama3.2_1B_flexrank_demo_gar_0.95',
 'physical_size_ratio': 0.947857468292631,
 'storage_reference': 'full FlexRank tensor payload',
 'decomposed_storage_size_ratio_vs_reference': 0.7003278109678037,
 'storage_vs_internal_delta': -0.24752965732482723,
 'deployed_eval_loss': 2.608842611312866,
 'reloaded_eval_loss': 2.608842611312866}

### Reload verification
Save the deployed model once more, reload it, and assert that evaluation is unchanged.

In [11]:
VERIFY_SAVE_DIR = SAVE_DIR.parent / f"{SAVE_DIR.name}_verify_pruned_{DEPLOY_SIZE_RATIO:g}"

shutil.rmtree(VERIFY_SAVE_DIR, ignore_errors=True)
loaded_model.save_pretrained(VERIFY_SAVE_DIR)

verified_model = FlexRankModel.from_pretrained(VERIFY_SAVE_DIR)
verified_metrics = evaluate_model(verified_model, splits.val, splits.collator, model_data.proc)

verified_loss = extract_eval_loss(verified_metrics)
deployed_loss = extract_eval_loss(deploy_metrics)
verified_impl_counts = odlinear_impl_counts(verified_model)
expected_gar_layers = sum(isinstance(module, ODLinear) for module in verified_model.modules()) if DEPLOY_MODE is DeployMode.GAR else 0
assert verified_impl_counts[ODImpl.GAR.value] == expected_gar_layers
assert abs(verified_loss - deployed_loss) < 1e-6

{
    "save_dir": str(VERIFY_SAVE_DIR),
    "deployed_eval_loss": deployed_loss,
    "reloaded_eval_loss": verified_loss,
    "reloaded_odlinear_impl_counts": verified_impl_counts,
}


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'save_dir': 'outputs/notebooks/flexrank_llama3.2_1B_flexrank_demo_verify_pruned_0.95',
 'deployed_eval_loss': 2.608842611312866,
 'reloaded_eval_loss': 2.608842611312866,
 'reloaded_odlinear_impl_counts': {'gar': 112}}

## Downstream evaluation with lm-eval

Evaluate the final reloaded, physically deployed GAR model (`verified_model`) on the same six multiple-choice tasks used in the model card. The default runs the complete task splits with one labeled demonstration per query (`num_fewshot=1`) and reports lm-eval’s raw accuracy (`acc`). Set `LM_EVAL_LIMIT` to a positive integer only for a quick smoke test; limited results are not comparable with the published scores.


In [ ]:
import torch
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM
from lm_eval.utils import make_table

LM_EVAL_TASKS = [
    "arc_challenge",
    "arc_easy",
    "hellaswag",
    "openbookqa",
    "piqa",
    "winogrande",
]
LM_EVAL_NUM_FEWSHOT = 1
LM_EVAL_BATCH_SIZE = 32
LM_EVAL_LIMIT = None  # Use a positive integer only for a smoke test.

if DEPLOY_MODE is not DeployMode.GAR:
    raise RuntimeError("This section evaluates the final GAR deployment; set DEPLOY_MODE to GAR and rerun the deployment cells.")
if verified_impl_counts.get(ODImpl.GAR.value, 0) != expected_gar_layers:
    raise RuntimeError("verified_model is not the fully GAR-deployed model produced above.")

lm_eval_model = HFLM(
    pretrained=verified_model,
    tokenizer=model_data.proc,
    trust_remote_code=True,
    batch_size=LM_EVAL_BATCH_SIZE,
)

with torch.inference_mode():
    gar_lm_eval_results = evaluator.simple_evaluate(
        model=lm_eval_model,
        tasks=LM_EVAL_TASKS,
        batch_size=LM_EVAL_BATCH_SIZE,
        num_fewshot=LM_EVAL_NUM_FEWSHOT,
        limit=LM_EVAL_LIMIT,
        bootstrap_iters=0,
        cache_requests=True,
        check_integrity=False,
        log_samples=False,
    )

if gar_lm_eval_results is None:
    raise RuntimeError("lm-eval did not return results.")


In [13]:
print(make_table(gar_lm_eval_results))

|    Tasks    |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-------------|------:|------|-----:|--------|---|-----:|---|------|
|arc_challenge|      1|none  |     1|acc     |↑  |0.3311|±  |   N/A|
|             |       |none  |     1|acc_norm|↑  |0.3592|±  |   N/A|
|arc_easy     |      1|none  |     1|acc     |↑  |0.6814|±  |   N/A|
|             |       |none  |     1|acc_norm|↑  |0.6616|±  |   N/A|
|hellaswag    |      1|none  |     1|acc     |↑  |0.4494|±  |   N/A|
|             |       |none  |     1|acc_norm|↑  |0.5922|±  |   N/A|
|openbookqa   |      1|none  |     1|acc     |↑  |0.2680|±  |   N/A|
|             |       |none  |     1|acc_norm|↑  |0.3700|±  |   N/A|
|piqa         |      1|none  |     1|acc     |↑  |0.7443|±  |   N/A|
|             |       |none  |     1|acc_norm|↑  |0.7416|±  |   N/A|
|winogrande   |      1|none  |     1|acc     |↑  |0.6022|±  |   N/A|

